# Bagged Shallow Decision Trees

This notebook evaluates a voting ensemble of shallow decision trees for identifying fake rental listings. It shares the data preparation approach with `Model.ipynb` so the models can be compared on the same features and folds.


## 1. Imports and configuration

Install dependencies if needed: `python -m pip install pandas scikit-learn matplotlib`, then install the warning-sign detectors from the repo root with `python -m pip install -e housing-fraud-nlp`.


In [ ]:
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import BaggingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, make_scorer
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

from housing_fraud_nlp import WarningSignFeatures

RANDOM_STATE = 42
CV_FOLDS = 5
MAX_DEPTH = 1
N_TREES = 5


## 2. Load data

The JSON contains one object per rental listing. The data file is expected in the project working directory.


In [ ]:
def load_listings(json_path=Path.cwd() / "data.json"):
    """Read rental listings JSON into a pandas DataFrame."""
    return pd.read_json(json_path, encoding="utf-8")

data = load_listings()
data.head()


## 3. Prepare features and target

Once the full dataset includes the boolean `Fake?` label, uncomment and run the preparation code. Longitude and latitude are mapped to the requested `lon` and `lat` feature names.


In [ ]:
# target_column = "Fake?"
interest_columns = ["price", "description", "currency", "price_period", "title", "lon", "lat"]

# data = data.rename(columns={"longitude": "lon", "latitude": "lat"})
# X = data[interest_columns].copy()
# y = data[target_column].astype(bool)


## 4. Define the bagging model

Each estimator is a decision stump (`max_depth=1`). Bootstrap samples make the trees differ, and their class probabilities are averaged for a vote. Preprocessing is part of the pipeline, so it is refit independently inside each CV fold.


In [ ]:
numeric_columns = ["price", "lon", "lat"]
categorical_columns = ["currency", "price_period"]

preprocess = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_columns),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_columns),
        ("description", TfidfVectorizer(max_features=300, min_df=2), "description"),
        ("title", TfidfVectorizer(max_features=100, min_df=1), "title"),
        ("warning_signs", WarningSignFeatures(), ["title", "description"]),
    ],
    remainder="drop",
)

base_tree = DecisionTreeClassifier(
    max_depth=MAX_DEPTH, class_weight="balanced", random_state=RANDOM_STATE
)
ensemble = BaggingClassifier(
    estimator=base_tree, n_estimators=N_TREES, max_samples=0.8,
    bootstrap=True, max_features=0.8, bootstrap_features=False,
    random_state=RANDOM_STATE, n_jobs=-1
)
model = Pipeline([("preprocess", preprocess), ("ensemble", ensemble)])


## 5. Five-fold cross-validation

This reports balanced accuracy and fake-class precision, recall, and F1 for each fold, plus mean and standard deviation. Five-fold stratification requires at least five listings in each class.


In [ ]:
# scoring = {
#     "balanced_accuracy": "balanced_accuracy",
#     "fake_precision": make_scorer(precision_score, pos_label=True, zero_division=0),
#     "fake_recall": make_scorer(recall_score, pos_label=True, zero_division=0),
#     "fake_f1": make_scorer(f1_score, pos_label=True, zero_division=0),
# }
# cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
# scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
# for metric in scoring:
#     fold_scores = scores[f"test_{metric}"]
#     print(f"{metric}: {fold_scores.round(3)} | mean={fold_scores.mean():.3f} +/- {fold_scores.std():.3f}")


## Notes for comparison

Use the same five-fold splits and scoring metrics in `Model.ipynb` and this notebook. The ensemble is intentionally small for the initial experiment; once labels are available, compare a larger number of estimators and `min_samples_leaf` values using cross-validation only. Keep the held-out test set untouched for final evaluation.
